In [ ]:
import requests
import pandas as pd
import time

In [ ]:
app_ids = [
    730,      # Counter-Strike 2
    570,      # Dota 2
    440,      # Team Fortress 2
    578080,   # PUBG
    271590,   # GTA V
    1172470,  # Apex Legends
    1085660,  # Destiny 2
    381210,   # Dead by Daylight
    1091500,  # Cyberpunk 2077
    1245620,  # ELDEN RING
    1938090,  # Call of Duty
    252490,   # Rust
    1599340,  # Lost Ark
    413150,   # Stardew Valley
    814380,   # Sekiro
    292030,   # The Witcher 3
    553850,   # HELLDIVERS 2
    236390,   # War Thunder
    394360,   # Hearts of Iron IV
    1086940   # Baldur's Gate 3
]

In [ ]:
url = f"https://store.steampowered.com/api/appdetails?appids={730}"
response = requests.get(url, timeout=10)
data = response.json()
print(data.keys())

In [ ]:
data['730']

In [ ]:
def get_game_metadata(app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        if not data[str(app_id)]["success"]:
            return None
        
        game = data[str(app_id)]["data"]
        
        return {
            "app_id": app_id,
            "name": game.get("name"),
            "type": game.get("type"),
            "is_free": game.get("is_free"),
            "release_date": game.get("release_date", {}).get("date"),
            "coming_soon": game.get("release_date", {}).get("coming_soon"),
            "price_usd": (
                game.get("price_overview", {}).get("final", None) / 100
                if game.get("price_overview") and game.get("price_overview", {}).get("final") is not None
                else 0 if game.get("is_free") else None
            ),
            "developers": ", ".join(game.get("developers", [])) if game.get("developers") else None,
            "publishers": ", ".join(game.get("publishers", [])) if game.get("publishers") else None,
            "genres": ", ".join([g["description"] for g in game.get("genres", [])]) if game.get("genres") else None,
            "categories": ", ".join([c["description"] for c in game.get("categories", [])]) if game.get("categories") else None,
            "dlc_count": len(game.get("dlc", [])) if game.get("dlc") else 0,
            "required_age": game.get("required_age"),
            "header_image": game.get("header_image"),
            "website": game.get("website")
        }
    
    except Exception as e:
        print(f"Error for app_id {app_id}: {e}")
        return None

In [ ]:
records = []

for app_id in app_ids:
    row = get_game_metadata(app_id)
    if row is not None:
        records.append(row)
    time.sleep(1)  # give the API some time

df_metadata = pd.DataFrame(records)
df_metadata.head()

In [ ]:
print(df_metadata.shape)
print(df_metadata.columns)
df_metadata.info()

In [ ]:
df_metadata.to_csv("../data/raw/steam_metadata_batch1.csv", index=False)
print("Saved first batch.")